In [1]:
import os, re

In [2]:
with open(r'D:\Projects\TwainToken\data\data_p1.txt', 'r', encoding='utf-8') as f:
    corpus = f.read()

### Test logic

In [3]:
token_pairs = {}

for i in range(len(corpus) - 1):
    t1, t2 = ord(corpus[i]), ord(corpus[i + 1])
    pair = (t1, t2)
    token_pairs[pair] = token_pairs.get(pair, 0) + 1

In [4]:
token_pairs = sorted(token_pairs.items(), key=lambda x: x[1], reverse=True)

In [5]:
print(token_pairs[:10])

[((101, 32), 32798), ((32, 116), 27944), ((116, 104), 25835), ((104, 101), 24020), ((32, 97), 22516), ((100, 32), 20445), ((115, 32), 18680), ((116, 32), 16786), ((97, 110), 16606), ((105, 110), 14398)]


In [6]:
tokens = list(corpus.encode('utf-8'))
tokens[:5]

[84, 104, 101, 32, 80]

In [7]:
max(tokens)

226

In [ ]:
def get_stats(tokens:list) -> dict:
    """
    returns {
    (token1, token2): count,
     ...
     }
    """

    token_pairs = {}
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        token_pairs[pair] = token_pairs.get(pair, 0) + 1
    
    return token_pairs


def merge_tokens(tokens:list, pair:tuple, new_id:int) -> list:
    new_tokens = []
    i = 0

    while i < len(tokens) - 1:
        if (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_id)
            i += 2
        
        else:
            new_tokens.append(tokens[i])
            i += 1

    if i == len(tokens) - 1:
        new_tokens.append(tokens[-1])

    return new_tokens


def train_bpe(corpus:str, num_merges:int) -> tuple:
    tokens = list(corpus.encode('utf-8'))
    token_counts = get_stats(tokens)
    merge_count = 0
    merge_table = {}

    while merge_count < num_merges:
        if not token_counts:
            break

        new_id = 256 + merge_count
        best_pair = max(token_counts, key=token_counts.get)
        tokens = merge_tokens(tokens, best_pair, new_id)
        merge_table[best_pair] = new_id
        token_counts = get_stats(tokens)
        merge_count += 1
        print(f'Merge {merge_count}/{num_merges}', end='\r')

    return tokens, merge_table

In [9]:
tks, table = train_bpe(corpus, 100)

In [ ]:
def encode_text(text:str, merge_table:dict) -> list:
    tokens = list(text.encode('utf-8'))

    for pair, new_id in merge_table.items():
        tokens = merge_tokens(tokens, pair, new_id)

    return tokens


def expand_token(token:int, reverse_table:dict) -> list:
    """
    Revert compressed token pairs back to their original byte sequences
    """
    if token < 256:
        return [token]

    left, right = reverse_table[token]
    
    return expand_token(left, reverse_table) + expand_token(right, reverse_table)


def decode_tokens(tokens:list, merge_table:dict) -> str:
    reverse_table = {v: k for k, v in merge_table.items()}
    decoded_tokens = []

    for token in tokens:
        if token < 256:
            decoded_tokens.append(token)

        elif token in reverse_table:
            decoded_tokens.extend(expand_token(token, reverse_table))

        else:
            decoded_tokens.append(ord('?'))

    return bytes(decoded_tokens).decode('utf-8', errors='replace')

In [22]:
text = "Hello, world!"
encoded = encode_text(text, table) 
decoded = decode_tokens(encoded, table)

encoded, decoded

([72, 294, 108, 111, 266, 119, 274, 108, 100, 33], 'Hello, world!')

In [23]:
text = "नमस्कार, जग!"
encoded = encode_text(text, table) 
decoded = decode_tokens(encoded, table)

encoded, decoded

([224,
  164,
  168,
  224,
  164,
  174,
  224,
  164,
  184,
  224,
  165,
  141,
  224,
  164,
  149,
  224,
  164,
  190,
  224,
  164,
  176,
  266,
  224,
  164,
  156,
  224,
  164,
  151,
  33],
 'नमस्कार, जग!')